In [ ]:
import os
import random
import numpy as np
import torch
import time
from statsmodels.tsa.seasonal import STL

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error
from sklearn.metrics.pairwise import haversine_distances
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

In [ ]:
os.makedirs('outputs', exist_ok=True)

In [ ]:
NODE_FEATURES = [
    'DIR_log',        
    'Month_Sin',      
    'Month_Cos', 
    'temp_mean',      
    'humidity_mean',
    'precip_sum',
]
TARGET_SOURCE_COL = 'DIR_log'  

#  Preprocessing 
df_raw = pd.read_csv('data/df_final.csv')
df_raw['Year_Month'] = pd.to_datetime(df_raw['Year_Month'])
df_raw = df_raw.sort_values(['Kab/Kota', 'Year_Month']).reset_index(drop=True)

df_raw['Month']          = df_raw['Year_Month'].dt.month
df_raw['Month_Sin']      = np.sin(2 * np.pi * df_raw['Month'] / 12)
df_raw['Month_Cos']      = np.cos(2 * np.pi * df_raw['Month'] / 12)

df_raw['target'] = df_raw.groupby('Kab/Kota')[TARGET_SOURCE_COL].shift(-1)

df_raw.dropna(inplace=True)

NODE_LIST = sorted(df_raw['Kab/Kota'].unique())
N_NODES   = len(NODE_LIST)

def kab_to_time_major(a, n_kab, t_per_kab):
    return a.reshape(n_kab, t_per_kab).T.reshape(-1)

timestamps_pd = pd.to_datetime(sorted(df_raw['Year_Month'].unique()))
train_months  = timestamps_pd[timestamps_pd < '2018-01-01']
valid_months  = timestamps_pd[(timestamps_pd >= '2018-01-01') & (timestamps_pd < '2022-01-01')]
test_months   = timestamps_pd[(timestamps_pd >= '2022-01-01') & (timestamps_pd < "2024-01-01")]

df_train = df_raw[df_raw['Year_Month'].isin(train_months)].dropna(subset=['target'])
df_valid  = df_raw[df_raw['Year_Month'].isin(valid_months)].dropna(subset=['target'])
df_test   = df_raw[df_raw['Year_Month'].isin(test_months)].dropna(subset=['target'])

# Scaler fit pada train saja
scaler_feat = StandardScaler()
X_train_tab = scaler_feat.fit_transform(df_train[NODE_FEATURES])
X_valid_tab  = scaler_feat.transform(df_valid[NODE_FEATURES])
X_test_tab   = scaler_feat.transform(df_test[NODE_FEATURES])

y_train_tab = df_train['target'].values
y_valid_tab  = df_valid['target'].values
y_test_tab   = df_test['target'].values

print(f'Node features ({len(NODE_FEATURES)}): {NODE_FEATURES}')
print(f'Train: {X_train_tab.shape} | Valid: {X_valid_tab.shape} | Test: {X_test_tab.shape}')
print(f'Kab/Kota: {N_NODES}')


In [ ]:
LAGS = 11   
df_lag = df_raw.copy()
lag_cols = []
for f in NODE_FEATURES:
    for L in range(1, LAGS + 1):
        col = f'{f}_lag{L}'
        df_lag[col] = df_lag.groupby('Kab/Kota')[f].shift(L)
        df_lag[col] = df_lag[col].fillna(df_lag[f])  
        lag_cols.append(col)

FEATURES_LAG = NODE_FEATURES + lag_cols  

df_train_lag = df_lag[df_lag['Year_Month'].isin(train_months)].dropna(subset=['target'])
df_valid_lag = df_lag[df_lag['Year_Month'].isin(valid_months)].dropna(subset=['target'])
df_test_lag  = df_lag[df_lag['Year_Month'].isin(test_months )].dropna(subset=['target'])

scaler_lag  = StandardScaler()
X_train_lag = scaler_lag.fit_transform(df_train_lag[FEATURES_LAG])
X_valid_lag = scaler_lag.transform(df_valid_lag[FEATURES_LAG])
X_test_lag  = scaler_lag.transform(df_test_lag[FEATURES_LAG])

assert X_train_lag.shape[0] == X_train_tab.shape[0], (X_train_lag.shape, X_train_tab.shape)
assert X_valid_lag.shape[0] == X_valid_tab.shape[0]
assert X_test_lag.shape[0]  == X_test_tab.shape[0]
print(f'Lag matrices  train {X_train_lag.shape} | valid {X_valid_lag.shape} | test {X_test_lag.shape}')
print(f'FEATURES_LAG = {len(NODE_FEATURES)} base + {len(lag_cols)} lag = {len(FEATURES_LAG)}')

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2)))

def eval_metrics(y_true, y_pred, label=''):
    r  = rmse(y_true, y_pred)
    m  = mean_absolute_error(y_true, y_pred)
    if label:
        print(f'  {label:<25} RMSE={r:.6f}  MAE={m:.6f}')
    return {'RMSE': r, 'MAE': m}

print('Metric functions defined: rmse, eval_metrics')


#### Train Multi Seed

In [ ]:
SEEDS = [42, 123, 456, 789, 2024]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

In [ ]:
def kab_to_time_major(a, n_kab, t_per_kab):
    return a.reshape(n_kab, t_per_kab).T.reshape(-1)

In [ ]:
all_baseline_results = {}

In [ ]:
# LSTM 

WINDOW     = 12
BATCH_SIZE = 16
LSTM_HIDDEN = 64
LSTM_LAYERS = 1
LSTM_DROPOUT = 0.2
LSTM_EPOCHS  = 200
LSTM_PATIENCE = 20
DEVICE_LSTM = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Pivot ke (T, N, F) seperti STGNN
timestamps_all = sorted(df_raw['Year_Month'].unique())
T = len(timestamps_all)
node2idx_fc = {n: i for i, n in enumerate(NODE_LIST)}

X_raw_seq = np.zeros((T, N_NODES, len(NODE_FEATURES)), dtype=np.float32)
y_raw_seq = np.zeros((T, N_NODES), dtype=np.float32)

for t_idx, ts in enumerate(timestamps_all):
    df_t = df_raw[df_raw['Year_Month'] == ts]
    for _, row in df_t.iterrows():
        n_idx = node2idx_fc.get(row['Kab/Kota'])
        if n_idx is not None:
            X_raw_seq[t_idx, n_idx, :] = row[NODE_FEATURES].values
            y_raw_seq[t_idx, n_idx]    = row['target'] if pd.notna(row['target']) else 0.0

X_seq_sc = scaler_feat.transform(X_raw_seq.reshape(-1, len(NODE_FEATURES))).reshape(T, N_NODES, -1)

timestamps_pd_all = pd.to_datetime(timestamps_all)
tr_mask = timestamps_pd_all < '2018-01-01'
vl_mask = (timestamps_pd_all >= '2018-01-01') & (timestamps_pd_all < '2022-01-01')
te_mask = (timestamps_pd_all >= '2022-01-01') & (timestamps_pd_all < '2024-01-01')

X_tr_sc = X_seq_sc[tr_mask]; y_tr_sc = y_raw_seq[tr_mask]
X_vl_sc = X_seq_sc[vl_mask]; y_vl_sc = y_raw_seq[vl_mask]
X_te_sc = X_seq_sc[te_mask]; y_te_sc = y_raw_seq[te_mask]


class SeqDataset(Dataset):
    def __init__(self, X, y, window=12):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.w = window
    def __len__(self):
        return len(self.X) - self.w + 1  
    def __getitem__(self, i):
        return self.X[i:i+self.w], self.y[i+self.w-1]


class LSTMModel(nn.Module):
    def __init__(self, n_features, hidden=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, n_layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden, 1)

    def forward(self, x):
        batch, window, N, F = x.shape
        x_flat = x.permute(0, 2, 1, 3).reshape(batch * N, window, F)
        out, _ = self.lstm(x_flat)
        pred   = self.fc(out[:, -1, :]).squeeze(-1)
        return pred.reshape(batch, N)


tr_ds = SeqDataset(X_tr_sc, y_tr_sc, WINDOW)
vl_ds = SeqDataset(X_vl_sc, y_vl_sc, WINDOW)
te_ds = SeqDataset(X_te_sc, y_te_sc, WINDOW)
tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=False)
vl_dl = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False)
te_dl = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
for seed in SEEDS:
    print(f'\n{"="*70}')
    print(f'SEED = {seed}')
    print(f'{"="*70}')
    seed_results = {}
 
    # CatBoost 
    print(f'\n  [Seed {seed}] Training CatBoost...')
    set_seed(seed)
 
    cb_model = CatBoostRegressor(
        iterations    = 2000,
        learning_rate = 0.05,
        depth         = 4,
        loss_function = 'RMSE',
        eval_metric   = 'RMSE',
        random_seed   = seed,
        verbose       = 100,
        early_stopping_rounds = 50,
    )

    _t0 = time.perf_counter()
    cb_model.fit(X_train_tab, y_train_tab, eval_set=(X_valid_tab, y_valid_tab), use_best_model=True)
    seed_results['CatBoost_train_time_sec'] = time.perf_counter() - _t0
    print(f"    CatBoost done. Time: {seed_results['CatBoost_train_time_sec']:.2f} s")
    
    pred_cb_train = cb_model.predict(X_train_tab)
    pred_cb_valid = cb_model.predict(X_valid_tab)
    pred_cb_test  = cb_model.predict(X_test_tab)
    print(f'    CatBoost done.')
 
    # SVM 
    print(f'\n  [Seed {seed}] Training SVM...')
    set_seed(seed)
 
    svm_model = SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale')

    _t0 = time.perf_counter()
    svm_model.fit(X_train_tab, y_train_tab)
    seed_results['SVM_train_time_sec'] = time.perf_counter() - _t0
    print(f"    SVM done. Time: {seed_results['SVM_train_time_sec']:.2f} s")
    
    pred_svm_train = svm_model.predict(X_train_tab)
    pred_svm_valid = svm_model.predict(X_valid_tab)
    pred_svm_test  = svm_model.predict(X_test_tab)
    print(f'    SVM done.')

    #  CatBoost-lag12 & SVM-lag12  (Group B: information-matched 12-month lag) 
    print(f'\n  [Seed {seed}] Training CatBoost-lag12...')
    set_seed(seed)
    cb_lag_model = CatBoostRegressor(
        iterations=2000, learning_rate=0.05, depth=4,
        loss_function='RMSE', eval_metric='RMSE',
        random_seed=seed, verbose=0, early_stopping_rounds=50,
    )
    cb_lag_model.fit(X_train_lag, y_train_tab,
                     eval_set=(X_valid_lag, y_valid_tab), use_best_model=True)
    pred_cblag_test = cb_lag_model.predict(X_test_lag)

    print(f'  [Seed {seed}] Training SVM-lag12...')
    set_seed(seed)
    svm_lag_model = SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale')  # sama spt SVM-A -> hanya fitur yg beda
    svm_lag_model.fit(X_train_lag, y_train_tab)
    pred_svmlag_test = svm_lag_model.predict(X_test_lag)
    print(f'    lag models done.')
 
    #  LSTM
    print(f'\n  [Seed {seed}] Training LSTM...')
    set_seed(seed)

    
    
    DEVICE_LSTM = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    lstm_model  = LSTMModel(len(NODE_FEATURES), LSTM_HIDDEN, LSTM_LAYERS,
                             LSTM_DROPOUT).to(DEVICE_LSTM)
    optimizer  = torch.optim.Adam(lstm_model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion   = nn.MSELoss()
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8, factor=0.5)
    lstm_path   = f'outputs/lstm_baseline_seed{seed}.pt'
 
    best_val, patience_c = np.inf, 0
    _t0 = time.perf_counter()
    for epoch in range(1, LSTM_EPOCHS + 1):
        lstm_model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE_LSTM), yb.to(DEVICE_LSTM)
            optimizer.zero_grad()
            loss = criterion(lstm_model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
            optimizer.step()
    
        lstm_model.eval()
        vl_sum, vl_n = 0.0, 0
        with torch.no_grad():
            for xb, yb in vl_dl:
                xb, yb = xb.to(DEVICE_LSTM), yb.to(DEVICE_LSTM)
                l = criterion(lstm_model(xb), yb).item()
                vl_sum += l * xb.size(0)
                vl_n   += xb.size(0)
        vl = vl_sum / vl_n
        scheduler.step(vl)
    
        if vl < best_val:
            best_val, patience_c = vl, 0
            torch.save(lstm_model.state_dict(), lstm_path)
        else:
            patience_c += 1
            if patience_c >= LSTM_PATIENCE:
                print(f'    LSTM early stop @ epoch {epoch}')
                break
 
    lstm_model.load_state_dict(torch.load(lstm_path, map_location=DEVICE_LSTM))
    lstm_model.eval()

    def collect_preds(dl, model, device):
        preds, tgts = [], []
        with torch.no_grad():
            for xb, yb in dl:
                preds.append(model(xb.to(device)).cpu().numpy())
                tgts.append(yb.numpy())
        return np.concatenate(preds), np.concatenate(tgts)

    pred_lstm_train_arr, tgt_lstm_train = collect_preds(tr_dl, lstm_model, DEVICE_LSTM)
    pred_lstm_valid_arr, tgt_lstm_valid = collect_preds(vl_dl, lstm_model, DEVICE_LSTM)
    pred_lstm_test_arr,  tgt_lstm_test  = collect_preds(te_dl, lstm_model, DEVICE_LSTM)
 
    lstm_train_al = pred_lstm_train_arr.flatten()
    lstm_valid_al = pred_lstm_valid_arr.flatten()
    lstm_test_al  = pred_lstm_test_arr.flatten()
    print(f'    LSTM done.')

    seed_results['LSTM_train_time_sec'] = time.perf_counter() - _t0
    print(f"    LSTM done. Time: {seed_results['LSTM_train_time_sec']:.2f} s")
     
    T_train_per_kab = len(y_train_tab) // N_NODES
    T_valid_per_kab = len(y_valid_tab) // N_NODES
    T_test_per_kab  = len(y_test_tab)  // N_NODES
    offset          = (WINDOW - 1) * N_NODES
 
    cb_train_al  = kab_to_time_major(pred_cb_train,  N_NODES, T_train_per_kab)[offset:]
    cb_valid_al  = kab_to_time_major(pred_cb_valid,  N_NODES, T_valid_per_kab)[offset:]
    cb_test_al   = kab_to_time_major(pred_cb_test,   N_NODES, T_test_per_kab) [offset:]
 
    svm_train_al = kab_to_time_major(pred_svm_train, N_NODES, T_train_per_kab)[offset:]
    svm_valid_al = kab_to_time_major(pred_svm_valid, N_NODES, T_valid_per_kab)[offset:]
    svm_test_al  = kab_to_time_major(pred_svm_test,  N_NODES, T_test_per_kab) [offset:]
    cblag_test_al  = kab_to_time_major(pred_cblag_test,  N_NODES, T_test_per_kab) [offset:]
    svmlag_test_al = kab_to_time_major(pred_svmlag_test, N_NODES, T_test_per_kab) [offset:]
 
    y_train_al   = kab_to_time_major(y_train_tab,    N_NODES, T_train_per_kab)[offset:]
    y_valid_al   = kab_to_time_major(y_valid_tab,    N_NODES, T_valid_per_kab)[offset:]
    y_test_al    = kab_to_time_major(y_test_tab,     N_NODES, T_test_per_kab) [offset:]

    _drop1 = N_NODES
    cb_test_al   = cb_test_al[_drop1:]
    svm_test_al  = svm_test_al[_drop1:]
    lstm_test_al = lstm_test_al[_drop1:]
    y_test_al    = y_test_al[_drop1:]
    cblag_test_al  = cblag_test_al [_drop1:]
    svmlag_test_al = svmlag_test_al[_drop1:]
 
    # Sanity check
    assert len(cb_train_al) == len(lstm_train_al), \
        f'[Seed {seed}] MISMATCH train: cb={len(cb_train_al)}, lstm={len(lstm_train_al)}'
    assert len(cb_test_al) == len(lstm_test_al), \
        f'[Seed {seed}] MISMATCH test: cb={len(cb_test_al)}, lstm={len(lstm_test_al)}'
    print(f'\n  Aligned samples — train: {len(cb_train_al)} | '
          f'valid: {len(cb_valid_al)} | test: {len(cb_test_al)}')
 
    # Ensemble RF 
    print(f'\n  [Seed {seed}] Training Ensemble RF meta-learner...')
    set_seed(seed)
 
    X_meta_tv = np.vstack([
        np.column_stack([cb_train_al, svm_train_al, lstm_train_al]),
        np.column_stack([cb_valid_al, svm_valid_al, lstm_valid_al]),
    ])
    y_meta_tv = np.concatenate([y_train_al, y_valid_al])
 
    X_meta_test = np.column_stack([cb_test_al, svm_test_al, lstm_test_al])
 
    rf_meta = RandomForestRegressor(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        random_state=42, n_jobs=-1,
    )

    _t0 = time.perf_counter()
    rf_meta.fit(X_meta_tv, y_meta_tv)
    seed_results['Ensemble_RF_train_time_sec'] = time.perf_counter() - _t0
    print(f"    Ensemble done. Time: {seed_results['Ensemble_RF_train_time_sec']:.2f} s")
    ens_test = rf_meta.predict(X_meta_test)
    print(f'    Ensemble RF done. Input: {X_meta_test.shape}')

    seed_results['CatBoost']    = eval_metrics(y_test_al, cb_test_al,   f'CatBoost  seed={seed}')
    seed_results['SVM']         = eval_metrics(y_test_al, svm_test_al,  f'SVM       seed={seed}')
    seed_results['LSTM']        = eval_metrics(y_test_al, lstm_test_al, f'LSTM      seed={seed}')
    seed_results['Ensemble_RF'] = eval_metrics(y_test_al, ens_test,     f'Ensemble  seed={seed}')
    seed_results['CatBoost_lag12'] = eval_metrics(y_test_al, cblag_test_al,  f'CatBoost-lag12 seed={seed}')
    seed_results['SVM_lag12']      = eval_metrics(y_test_al, svmlag_test_al, f'SVM-lag12      seed={seed}')

    def _to_dir(a):
        return np.expm1(np.asarray(a, float))         
     
    def _dual(target_log, pred_log):
        t = np.asarray(target_log, float).ravel()
        p = np.asarray(pred_log,   float).ravel()
        t_dir, p_dir = _to_dir(t), _to_dir(p)
        return {
            'RMSE_log': float(np.sqrt(np.mean((t - p) ** 2))),
            'MAE_log' : float(np.mean(np.abs(t - p))),
            'RMSE_DIR': float(np.sqrt(np.mean((t_dir - p_dir) ** 2))),
            'MAE_DIR' : float(np.mean(np.abs(t_dir - p_dir))),
        }
     
    preds_this_seed = {
        'CatBoost'      : cb_test_al,
        'SVM'           : svm_test_al,
        'CatBoost_lag12': cblag_test_al,
        'SVM_lag12'     : svmlag_test_al,
        'LSTM'          : lstm_test_al,
        'Ensemble_RF'   : ens_test,
    }
     
    for name, pred_arr in preds_this_seed.items():
        seed_results[name + '_dual'] = _dual(y_test_al, pred_arr)
     
    np.savez(
        f'outputs/baseline_predictions_seed{seed}.npz',
        y_test_al=y_test_al,
        cb_test_al=cb_test_al,
        svm_test_al=svm_test_al,
        cblag_test_al=cblag_test_al,
        svmlag_test_al=svmlag_test_al,
        lstm_test_al=lstm_test_al,
        ens_test=ens_test,
    )
    print(f'  [Seed {seed}] Dual-scale metrics + prediction.')
    
    all_baseline_results[seed] = seed_results
    print(f'\n  Seed {seed} complete.')

In [ ]:
lstm_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
cb_trees    = cb_model.tree_count_
svm_support_vectors = svm_model.support_vectors_.shape[0]
rf_nodes    = sum(t.tree_.node_count for t in rf_meta.estimators_)

print(f'LSTM trainable parameters : {lstm_params:,}')
print(f'CatBoost trees            : {cb_trees:,}')
print(f'SVM support vectors       : {svm_support_vectors:,} / {len(X_train_tab):,} train samples')
print(f'RF meta-learner nodes     : {rf_nodes:,} (across {rf_meta.n_estimators} trees)')

In [ ]:
baselines        = ['CatBoost', 'SVM', 'LSTM', 'Ensemble_RF']
# Group A (current-month) + Group B (lag12) + Group C (deep) untuk tabel dual-scale:
baselines_dual   = ['CatBoost', 'SVM', 'CatBoost_lag12', 'SVM_lag12', 'LSTM', 'Ensemble_RF']
metrics_to_track = ['RMSE', 'MAE']
 
summary_rows = []
for model_name in baselines:
    row = {'Model': model_name}
    for metric in metrics_to_track:
        values   = [all_baseline_results[seed][model_name][metric] for seed in SEEDS]
        mean_val = np.mean(values)
        std_val  = np.std(values)
        row[f'{metric}_mean'] = mean_val
        row[f'{metric}_std']  = std_val
        row[f'{metric}_str']  = f'{mean_val:.4f} ± {std_val:.4f}'
    summary_rows.append(row)
 
baseline_summary_df = pd.DataFrame(summary_rows)
 
print('\n' + '=' * 90)
print(f'BASELINE ROBUSTNESS — {len(SEEDS)} seeds: {SEEDS}')
print('=' * 90)
display_df = baseline_summary_df[['Model'] + [f'{m}_str' for m in metrics_to_track]].copy()
display_df.columns = ['Model'] + metrics_to_track
print(display_df.to_string(index=False))
 
# Save
raw_rows = []
for seed in SEEDS:
    for model_name in baselines:
        row = {'Seed': seed, 'Model': model_name}
        for metric in metrics_to_track:
            row[metric] = all_baseline_results[seed][model_name][metric]
        raw_rows.append(row)
 
pd.DataFrame(raw_rows).to_csv('outputs/baseline_robustness_raw.csv', index=False)
baseline_summary_df.to_csv('outputs/baseline_robustness_summary.csv', index=False)
print('\nSaved → baseline_robustness_raw.csv & baseline_robustness_summary.csv')

In [ ]:
print("=" * 80)
print(f"TABEL 9 — Baseline, Dual Scale ({len(SEEDS)} seeds), mean ± std")
print("=" * 80)
print(f"{'Model':<16}{'RMSE_log':>12}{'MAE_log':>10}{'RMSE_DIR':>12}{'MAE_DIR':>10}")
print("-" * 80)
 
dual_summary_rows = []
for model_name in baselines_dual:                  # A: CatBoost/SVM | B: *_lag12 | C: LSTM/Ensemble
    key = model_name + '_dual'
    vals = {k: np.array([all_baseline_results[s][key][k] for s in SEEDS])
            for k in ['RMSE_log', 'MAE_log', 'RMSE_DIR', 'MAE_DIR']}
    row = {'Model': model_name}
    for k, v in vals.items():
        row[f'{k}_mean'] = v.mean()
        row[f'{k}_std']  = v.std()
    dual_summary_rows.append(row)
    print(f"{model_name:<16}"
          f"{vals['RMSE_log'].mean():>7.4f}±{vals['RMSE_log'].std():.4f} "
          f"{vals['MAE_log'].mean():>6.4f}±{vals['MAE_log'].std():.4f} "
          f"{vals['RMSE_DIR'].mean():>7.2f}±{vals['RMSE_DIR'].std():.2f} "
          f"{vals['MAE_DIR'].mean():>6.2f}±{vals['MAE_DIR'].std():.2f}")
 
df_baseline_dual = pd.DataFrame(dual_summary_rows)
df_baseline_dual.to_csv('outputs/table9_baseline_dual_scale.csv', index=False)
print("\n Salin kolom RMSE_DIR_mean / MAE_DIR_mean ke Tabel 9 di paper (baris baseline).")
print("   Saved: table9_baseline_dual_scale.csv")
 
baseline_preds = {}       
baseline_abs_err = {}   
 
for model_name, arr_key in [('CatBoost','cb_test_al'), ('SVM','svm_test_al'),
                             ('CatBoost_lag12','cblag_test_al'), ('SVM_lag12','svmlag_test_al'),
                             ('LSTM','lstm_test_al'), ('Ensemble_RF','ens_test')]:
    tgt_pool, pred_pool = [], []
    for seed in SEEDS:
        npz = np.load(f'outputs/baseline_predictions_seed{seed}.npz')
        tgt_pool.append(npz['y_test_al'])
        pred_pool.append(npz[arr_key])
    tgt_pool  = np.concatenate(tgt_pool)
    pred_pool = np.concatenate(pred_pool)
    baseline_preds[model_name]   = {'target': tgt_pool, 'pred': pred_pool}
    baseline_abs_err[model_name] = np.abs(pred_pool - tgt_pool)
 
np.savez(
    'outputs/baseline_pooled_for_wilcoxon.npz',
    **{f'{name}_target': d['target'] for name, d in baseline_preds.items()},
    **{f'{name}_pred':   d['pred']   for name, d in baseline_preds.items()},
)

#### Per-District Errors (n=119) 

In [ ]:
OUT_DIR = 'outputs'
N = N_NODES
assert N == 119, f'Expected 119 districts, got {N}'

base_arr_keys = {'CatBoost': 'cb_test_al', 'SVM': 'svm_test_al',
                 'CatBoost_lag12': 'cblag_test_al', 'SVM_lag12': 'svmlag_test_al',
                 'LSTM': 'lstm_test_al', 'Ensemble_RF': 'ens_test'}

frames = []
for seed in SEEDS:
    npz    = np.load(f'{OUT_DIR}/baseline_predictions_seed{seed}.npz')
    y      = npz['y_test_al']
    T_full = len(y) // N                    
    drop   = T_full - 12                    
    y2d    = y.reshape(T_full, N)[drop:]    
    for model, key in base_arr_keys.items():
        p2d = npz[key].reshape(T_full, N)[drop:]
        err = (p2d - y2d).ravel()           
        frames.append(pd.DataFrame({
            'Model'    : model,
            'Seed'     : seed,
            'Kab/Kota' : np.tile(NODE_LIST, 12),
            'month_idx': np.repeat(np.arange(12), N),   
            'abs_err'  : np.abs(err),
            'sq_err'   : err ** 2,
        }))

base_pd_df = pd.concat(frames, ignore_index=True)
base_pd_df.to_csv(f'{OUT_DIR}/baseline_per_district_raw.csv', index=False)
print(f'Saved baseline_per_district_raw.csv  rows={len(base_pd_df):,}  '
      f'(= 6 model × {len(SEEDS)} seed × 12 bulan × {N} distrik)')

_chk = (base_pd_df.groupby('Model')
        .agg(MAE=('abs_err', 'mean'),
             RMSE=('sq_err', lambda s: np.sqrt(s.mean())))
        .round(4))
print(_chk.to_string())

In [ ]:
STGNN_FULL = {
    'RMSE'      : (0.3266, 0.0026),   
    'MAE'       : (0.2450, 0.0018),   
}
 
# BUILD FINAL TABLE

temporal_flag = {
    'CatBoost'        : '—',
    'SVM'             : '—',
    'LSTM'            : '✓',
    'Ensemble_RF'     : '✓',
    'STGNN (proposed)': '✓',
}
spatial_flag = {
    'CatBoost'        : '—',
    'SVM'             : '—',
    'LSTM'            : '—',
    'Ensemble_RF'     : '—',
    'STGNN (proposed)': '✓',
}
 
final_rows = []
 
# Baseline rows
for model_name in baselines:
    row = {
        'Model'   : model_name,
        'Temporal': temporal_flag[model_name],
        'Spatial' : spatial_flag[model_name],
    }
    for metric in metrics_to_track:
        val = baseline_summary_df[
            baseline_summary_df['Model'] == model_name
        ][f'{metric}_str'].iloc[0]
        row[metric] = val
    final_rows.append(row)
 
# STGNN row (hardcoded) 
stgnn_row = {
    'Model'   : 'STGNN (proposed)',
    'Temporal': temporal_flag['STGNN (proposed)'],
    'Spatial' : spatial_flag['STGNN (proposed)'],
}
for metric, (mean_v, std_v) in STGNN_FULL.items():
    stgnn_row[metric] = f'{mean_v:.4f} ± {std_v:.4f}' if mean_v is not None else 'N/A'
 
final_rows.append(stgnn_row)
final_df = pd.DataFrame(final_rows).set_index('Model')
 
# PRINT & SAVE
print('\n' + '=' * 95)
print('  TABLE A — STGNN vs Baseline (5 seeds: [42, 123, 456, 789, 2024], mean ± std)')
print('  Test Set: 119 kabupaten/kota Pulau Jawa')
print('=' * 95)
print(final_df.to_string())
print('=' * 95)
 
final_df.to_csv('outputs/table_A_final_multiseed.csv')
print('\nSaved → table_A_final_multiseed.csv')
 
# IMPROVEMENT ANALYSIS 
print('\n' + '=' * 70)
print('IMPROVEMENT STGNN vs BASELINE (berdasarkan RMSE mean)')
print('=' * 70)
 
stgnn_rmse_mean = STGNN_FULL['RMSE'][0]
 
for model_name in baselines:
    baseline_rmse = baseline_summary_df[
        baseline_summary_df['Model'] == model_name
    ]['RMSE_mean'].iloc[0]
    
    delta = baseline_rmse - stgnn_rmse_mean
    pct   = delta / baseline_rmse * 100
    sign  = 'better' if delta > 0 else 'worse'
    print(f'  vs {model_name:<15}: ΔRMSE = {delta:+.4f}  ({pct:+.1f}%)  → STGNN {sign}')